In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import subprocess
import sys

# Install ultralytics (YOLOv8)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])

print("✓ Dependencies installed successfully!")

✓ Dependencies installed successfully!


In [2]:
from ultralytics import YOLO
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


In [3]:
# Dataset path
dataset_path = Path("Autonomous parking system.v2i.yolov8")
data_yaml = dataset_path / "data.yaml"

# Verify dataset exists
if dataset_path.exists():
    print(f"✓ Dataset found at: {dataset_path.absolute()}")
    
    # List dataset structure
    print("\nDataset Structure:")
    for split in ['train', 'valid', 'test']:
        split_path = dataset_path / split
        if split_path.exists():
            images = list((split_path / 'images').glob('*'))
            labels = list((split_path / 'labels').glob('*'))
            print(f"  {split}: {len(images)} images, {len(labels)} labels")
else:
    print(f"✗ Dataset not found at: {dataset_path}")

# Check if data.yaml exists
if data_yaml.exists():
    print(f"\n✓ data.yaml found")
    with open(data_yaml, 'r') as f:
        print("data.yaml content:")
        print(f.read())
else:
    print(f"✗ data.yaml not found at: {data_yaml}")

✓ Dataset found at: c:\Users\Vinay Kasana\OneDrive\Desktop\minor\Autonomous-Parking-System\Autonomous parking system.v2i.yolov8

Dataset Structure:
  train: 3321 images, 3321 labels
  valid: 318 images, 318 labels
  test: 157 images, 157 labels

✓ data.yaml found
data.yaml content:
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 4
names: ['Car', 'Space Available', 'busy', 'free']

roboflow:
  workspace: vinay-kasana
  project: autonomous-parking-system
  version: 2
  license: CC BY 4.0
  url: https://universe.roboflow.com/vinay-kasana/autonomous-parking-system/dataset/2


In [6]:
print("Loading pretrained YOLOv8 model...")
model = YOLO("yolov8n.pt")  # medium model - balanced between speed and accuracy
print("✓ Model loaded successfully!")

# Train the model
print("\nStarting training...")
print("This may take several minutes depending on your hardware...\n")

results = model.train(
    data=str(data_yaml),           # path to data.yaml
    epochs=1,                      # number of epochs
    imgsz=640,                      # image size
    batch=8,                        # batch size (adjust based on GPU memory)
    patience=10,                    # early stopping patience
    device="cpu",                       # GPU device (0 for first GPU, CPU if no GPU)
    project="parking_detection",    # project directory
    name="yolov8m_parking",         # experiment name
    exist_ok=False,                 # don't overwrite existing experiment
    save=True,                      # save training checkpoints
    save_period=10,                 # save every N epochs
    verbose=True,                   # verbose output
    pretrained=True,                # use pretrained weights
    optimizer="AdamW",                # optimizer
    seed=42                         # for reproducibility
)

print("\n✓ Training completed!")
print(f"Results saved to: parking_detection/yolov8m_parking")

Loading pretrained YOLOv8 model...
✓ Model loaded successfully!

Starting training...
This may take several minutes depending on your hardware...

New https://pypi.org/project/ultralytics/8.4.19 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.10.11 torch-2.10.0+cpu CPU (12th Gen Intel Core i5-1235U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Autonomous parking system.v2i.yolov8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line

In [11]:
# Load the best trained model
best_model_path = Path(r"C:\\Users\\Vinay Kasana\\OneDrive\\Desktop\\minor\\Autonomous-Parking-System\\runs\\detect\\parking_detection\\yolov8m_parking3\\weights\\best.pt")
if best_model_path.exists():
    print(f"Loading best model from: {best_model_path}")
    best_model = YOLO(str(best_model_path))
    
    # Validate on the validation set
    print("\nValidating on validation set...")
    val_results = best_model.val()
    
    print("\n" + "="*50)
    print("VALIDATION METRICS")
    print("="*50)
    print(f"mAP50: {val_results.box.map50:.4f}")
    print(f"mAP50-95: {val_results.box.map:.4f}")
    print(f"Precision: {val_results.box.mp:.4f}")
    print(f"Recall: {val_results.box.mr:.4f}")
else:
    print(f"Best model not found at: {best_model_path}")
    print("Make sure training completed successfully.")

Loading best model from: C:\Users\Vinay Kasana\OneDrive\Desktop\minor\Autonomous-Parking-System\runs\detect\parking_detection\yolov8m_parking3\weights\best.pt

Validating on validation set...
Ultralytics 8.4.14  Python-3.10.11 torch-2.10.0+cpu CPU (12th Gen Intel Core i5-1235U)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 1.60.3 ms, read: 18.57.2 MB/s, size: 38.3 KB)
val: Scanning C:\Users\Vinay Kasana\OneDrive\Desktop\minor\Autonomous-Parking-System\Autonomous parking system.v2i.yolov8\valid\labels.cache... 318 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 318/318 89.8Kit/s 0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 4006, len(boxes) = 8975. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R    

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dash 3.0.4 requires plotly>=5.0.0, which is not installed.
fastapi 0.115.11 requires pydantic!=1.8,!=1.8.1,!=2.0.0,!=2.0.1,!=2.1.0,<3.0.0,>=1.7.4, which is not installed.
groq 0.25.0 requires httpx<1,>=0.23.0, which is not installed.
groq 0.25.0 requires pydantic<3,>=1.9.0, which is not installed.
mediapipe 0.10.21 requires protobuf<5,>=4.25.3, which is not i

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 70% ━━━━━━━━──── 14/20 6.5s/it 1:31<38.9ssWARNING NMS time limit 2.800s exceeded
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 75% ━━━━━━━━━─── 15/20 6.8s/it 1:39<34.1sWARNING NMS time limit 2.800s exceeded
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 85% ━━━━━━━━━━── 17/20 7.5s/it 1:55<22.4sWARNING NMS time limit 2.800s exceeded
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 90% ━━━━━━━━━━╸─ 18/20 8.0s/it 2:05<16.1sWARNING NMS time limit 2.800s exceeded
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 7.4s/it 2:289.0s
                   all        318       8975      0.583      0.548      0.532      0.314
                   Car        172       2303      0.617      0.614      0.532      0.321
       Spa

In [13]:
from pathlib import Path
import random

# Get validation images
valid_images_path = Path(r"C:\Users\Vinay Kasana\OneDrive\Desktop\minor\Autonomous-Parking-System\Autonomous parking system.v2i.yolov8\valid\images")

if valid_images_path.exists():
    image_files = list(valid_images_path.glob("*.jpg")) + list(valid_images_path.glob("*.png"))
    
    if image_files:
        # Randomly sample up to 50 images
        num_samples = min(50, len(image_files))
        sampled_images = random.sample(image_files, num_samples)
        
        print(f"Running inference on {num_samples} randomly selected validation images...\n")
        
        # Adjust grid size as you like; here 5x10
        rows, cols = 5, 10
        fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
        axes = axes.flatten()
        
        for idx, img_path in enumerate(sampled_images):
            if idx >= rows * cols:
                break
            
            print(f"Processing: {img_path.name}")
            
            results = best_model.predict(source=str(img_path), conf=0.5, verbose=False)
            result = results[0]
            annotated_img = result.plot()
            
            axes[idx].imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
            axes[idx].set_title(f"Detections: {len(result.boxes)}", fontsize=8)
            axes[idx].axis("off")
        
        # Turn off any unused axes
        for j in range(idx + 1, rows * cols):
            axes[j].axis("off")
        
        plt.tight_layout()
        plt.savefig("sample_predictions_50.png", dpi=100, bbox_inches='tight')
        plt.show()
        
        print("\n✓ Sample predictions saved to: sample_predictions_50.png")
    else:
        print("No validation images found!")
else:
    print(f"Validation images directory not found: {valid_images_path}")


Running inference on 50 randomly selected validation images...

Processing: 1633298608_1-p-foto-mashini-vo-dvore-doma-foto-1_jpg.rf.aa4ba8eacef054ebcc3c53dfb6f669b4.jpg
Processing: 2016-01-18_1440_jpg.rf.7529f2cec1ced6b7f48f90e8745a7ea2.jpg
Processing: vlcsnap-2023-04-07-00h37m38s585_png_jpg.rf.44dd5b5dc8b61d01ef8a543bb0b93edd.jpg
Processing: VID-20231214-WA0011_mp4-168_jpg.rf.94a9c6e77dc50a0faa08e0b530cb0548.jpg
Processing: WhatsApp-Video-2023-09-14-at-14_32_36_mp4-56_jpg.rf.e8fa3dc15f85b1cf4e3a3ae5d6961f29.jpg
Processing: WhatsApp-Video-2023-09-14-at-14_20_14_mp4-14_jpg.rf.e2740b6bcafae39e5982828d06bdca0d.jpg
Processing: WhatsApp-Video-2023-09-14-at-14_20_14_mp4-154_jpg.rf.c5ee4a617748ccfb55f1205579094d70.jpg
Processing: WhatsApp-Video-2023-09-14-at-14_20_14_mp4-1059_jpg.rf.b9a249a9386aeb7284a430d0cc8c9c41.jpg
Processing: WhatsApp-Video-2023-09-14-at-14_19_54_mp4-14_jpg.rf.7f95468969f6faac6d5ef074bd09d979.jpg
Processing: GOPR6596_JPG_jpg.rf.9a1c0401297f9a5b7527ad6c312b4efd.jpg
Proces

<Figure size 3000x1500 with 50 Axes>


✓ Sample predictions saved to: sample_predictions_50.png


In [14]:
!pip install ultralytics opencv-python

import cv2
from ultralytics import YOLO


Defaulting to user installation because normal site-packages is not writeable


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


In [15]:
model = YOLO(r"C:\\Users\\Vinay Kasana\\OneDrive\\Desktop\\minor\\Autonomous-Parking-System\\runs\\detect\\parking_detection\\yolov8m_parking3\\weights\\best.pt")  # or yolov8n.pt etc.


In [ ]:
video_path = "/kaggle/input/datasets/vinay04kasana/car-video/13119394_3840_2160_25fps.mp4"
cap = cv2.VideoCapture(video_path)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("output.mp4", fourcc, 30.0,
                      (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                       int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    results = model(frame)          # run YOLO on this frame
    annotated = results[0].plot()   # draw boxes, labels, scores [web:18][web:24]

    out.write(annotated)            # save to output video

cap.release()
out.release()
cv2.destroyAllWindows()


In [6]:
# Copy-paste this EXACT cell
import sys
python_path = sys.executable
print("Installing to:", python_path)

!"{python_path}" -m pip install mlflow ultralytics --upgrade


Installing to: c:\Users\Vinay Kasana\AppData\Local\Programs\Python\Python310\python.exe


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



     ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
     --------------------------------------- 0.1/10.2 MB 871.5 kB/s eta 0:00:12
     --------------------------------------- 0.1/10.2 MB 871.5 kB/s eta 0:00:12
      -------------------------------------- 0.2/10.2 MB 958.4 kB/s eta 0:00:11
      -------------------------------------- 0.2/10.2 MB 958.4 kB/s eta 0:00:11
     - ------------------------------------- 0.3/10.2 MB 820.5 kB/s eta 0:00:13
     - ------------------------------------- 0.3/10.2 MB 984.6 kB/s eta 0:00:10
     - ------------------------------------- 0.4/10.2 MB 908.0 kB/s eta 0:00:11
     - -------------------------------------- 0.4/10.2 MB 1.0 MB/s eta 0:00:10
     - ------------------------------------- 0.5/10.2 MB 972.0 kB/s eta 0:00:10
     - ------------------------------------- 0.5/10.2 MB 972.0 k

In [8]:
import sys
python_path = sys.executable
print("Installing to:", python_path)

# --user installs to your user folder (no admin needed)
!"{python_path}" -m pip install mlflow ultralytics --user --upgrade


Installing to: c:\Users\Vinay Kasana\AppData\Local\Programs\Python\Python310\python.exe



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:

import mlflow

# If you have a tracking server, set it here.
# Otherwise this uses a local ./mlruns directory.
# mlflow.set_tracking_uri("http://127.0.0.1:5001")

EXPERIMENT_NAME = "autonomous_parking_yolo"
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id)


PermissionError: [WinError 5] Access is denied: 'C:\\Users\\Vinay%20Kasana'

In [ ]:
from ultralytics import YOLO
import time, os
import numpy as np  # ADD THIS

MODEL_NAME = "/kaggle/working/runs/detect/parking_detection/yolov8m_parking/weights/best.pt"
EPOCHS = 2
IMGSZ = 640
BATCH = 16
DEVICE = 0

model = YOLO(MODEL_NAME)

with mlflow.start_run(run_name="yolo_parking_v1") as run:
    run_id = run.info.run_id
    print("MLflow run_id:", run_id)

    # 1. Log hyperparameters
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("imgsz", IMGSZ)
    mlflow.log_param("batch", BATCH)
    mlflow.log_param("device", DEVICE)
    mlflow.log_param("data_yaml", data_yaml)

    # 2. Train YOLO
    t0 = time.time()
    results = model.train(
        data=data_yaml,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        name=f"parking_{run_id[:8]}",
        project="runs/detect",
        amp=True,
        cache=True,
    )
    train_time = time.time() - t0
    mlflow.log_metric("train_time_sec", train_time)

    # 3. Validate on val set
    val_results = model.val()
    mlflow.log_metric("map50", float(val_results.box.map50))
    mlflow.log_metric("map50_95", float(val_results.box.map))

    # FIX: Extract and aggregate per-class metrics
    precision_arr = val_results.box.p
    recall_arr = val_results.box.r
    mlflow.log_metric("precision_mean", float(np.mean(precision_arr)))
    mlflow.log_metric("recall_mean", float(np.mean(recall_arr)))

    # 4. Log artifacts
    run_dir = results.save_dir
    weights_best = os.path.join(run_dir, "weights", "best.pt")
    results_png = os.path.join(run_dir, "results.png")
    cm_png = os.path.join(run_dir, "confusion_matrix.png")
    val_pred_png = os.path.join(run_dir, "val_batch0_pred.jpg")

    for path, subdir in [
        (weights_best, "weights"),
        (results_png, "plots"),
        (cm_png, "plots"),
        (val_pred_png, "plots"),
    ]:
        if os.path.exists(path):
            mlflow.log_artifact(path, artifact_path=subdir)

    print("Artifacts logged. YOLO run dir:", run_dir)
    BEST_WEIGHTS_PATH = weights_best


In [ ]:
import mlflow.pyfunc
import pandas as pd
import cv2
from ultralytics import YOLO

class YOLOParkingModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        weights_path = context.artifacts["yolo_weights"]
        self.model = YOLO(weights_path)

    def predict(self, context, model_input: pd.DataFrame):
        # Expect a DataFrame with 'image_path' column
        paths = model_input["image_path"].tolist()
        outputs = []

        for p in paths:
            img = cv2.imread(p)
            if img is None:
                outputs.append({"image_path": p, "detections": [], "error": "read_failed"})
                continue

            results = self.model(img)
            dets = []
            for box, cls, conf in zip(results[0].boxes.xyxy.cpu().numpy(),
                                      results[0].boxes.cls.cpu().numpy(),
                                      results[0].boxes.conf.cpu().numpy()):
                x1, y1, x2, y2 = box.tolist()
                dets.append(
                    {"x1": float(x1), "y1": float(y1),
                     "x2": float(x2), "y2": float(y2),
                     "cls": int(cls), "conf": float(conf)}
                )
            outputs.append({"image_path": p, "detections": dets})

        return outputs

# Log + register the model using the best weights path from training cell
with mlflow.start_run(run_name="yolo_parking_register") as run:
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=YOLOParkingModel(),
        artifacts={"yolo_weights": BEST_WEIGHTS_PATH},
        registered_model_name="autonomous_parking_yolo",
    )
    print("Model registered as 'autonomous_parking_yolo'")


In [ ]:
!mlflow models serve -m "models:/autonomous_parking_yolo/1" -p 5000 --no-conda
